# Calibration and decision-curve analysis

Exercise notebook for the lesson **[Calibration and decision-curve analysis](https://ramkumarmd.com/medical-ai/machine-learning/calibration-and-decision-curves)** in the *Machine learning* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Calibration asks whether predicted risks match what actually happens; decision-curve analysis asks whether using the model would lead to better decisions than treating everyone or no one. When a model moves to a new hospital, it often keeps its ranking ability (AUROC) but its risks become too high or too low. Recalibration with a small local sample can fix this, and net benefit shows whether the model is worth using at the thresholds that matter clinically.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Why calibration fails at new sites

Develop at one synthetic site; apply to another with lower baseline risk.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

predictors = ["age", "bmi", "fasting_glucose", "family_history", "hypertension"]

def simulate(n, baseline, seed):
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({
        "age": rng.integers(30, 75, n),
        "bmi": rng.normal(27, 4.5, n).round(1),
        "fasting_glucose": rng.normal(100, 10, n).round(),
        "family_history": rng.binomial(1, 0.3, n),
        "hypertension": rng.binomial(1, 0.35, n),
    })
    logit = (baseline + 0.03 * d.age + 0.10 * d.bmi + 0.11 * d.fasting_glucose
             + 0.6 * d.family_history + 0.4 * d.hypertension)
    d["diabetes_5y"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return d

development = simulate(5000, baseline=-17.5, seed=5)
new_site = simulate(5000, baseline=-18.5, seed=6)        # lower baseline risk
model = LogisticRegression(penalty=None, max_iter=1000)
model.fit(development[predictors], development.diabetes_5y)

p_new = model.predict_proba(new_site[predictors])[:, 1]
print(f"New site AUROC {roc_auc_score(new_site.diabetes_5y, p_new):.3f}")
print(f"Mean predicted {p_new.mean():.1%}, observed {new_site.diabetes_5y.mean():.1%}")

groups = pd.qcut(p_new, 5, labels=False)
table = pd.DataFrame({"predicted": p_new, "observed": new_site.diabetes_5y}).groupby(groups).mean()
print((table * 100).round(1).rename_axis("risk fifth"))

# New site AUROC 0.806
# Mean predicted 18.9%, observed 9.4%
#             predicted  observed
# risk fifth
# 0                 2.6       1.0
# 1                 6.8       2.8
# 2                12.8       5.6
# 3                23.0       9.9
# 4                49.3      27.8

### Recalibrating with local data

Continuing from the previous block: recalibrate on 2,000 patients from the new site, evaluate on the other 3,000.

In [ ]:
recal_part, eval_part = new_site.iloc[:2000], new_site.iloc[2000:]

def log_odds(d):
    p = model.predict_proba(d[predictors])[:, 1]
    return np.log(p / (1 - p)).reshape(-1, 1)

recalibrator = LogisticRegression(penalty=None).fit(log_odds(recal_part), recal_part.diabetes_5y)
p_before = model.predict_proba(eval_part[predictors])[:, 1]
p_after = recalibrator.predict_proba(log_odds(eval_part))[:, 1]
print(f"Recalibration: slope {recalibrator.coef_[0][0]:.2f}, "
      f"intercept {recalibrator.intercept_[0]:.2f}")
print(f"Observed {eval_part.diabetes_5y.mean():.1%}; mean predicted before "
      f"{p_before.mean():.1%}, after {p_after.mean():.1%}")
print(f"AUROC before {roc_auc_score(eval_part.diabetes_5y, p_before):.3f}, "
      f"after {roc_auc_score(eval_part.diabetes_5y, p_after):.3f}")

# Recalibration: slope 0.93, intercept -1.01
# Observed 9.1%; mean predicted before 18.7%, after 9.6%
# AUROC before 0.812, after 0.812

### Net benefit: is the model worth using?

Continuing from the previous block: net benefit of the model before and after recalibration.

In [ ]:
y = eval_part.diabetes_5y.to_numpy()

def net_benefit(p, threshold):
    flagged = p >= threshold
    tp = np.sum(flagged & (y == 1)) / len(y)
    fp = np.sum(flagged & (y == 0)) / len(y)
    return tp - fp * threshold / (1 - threshold)

prevalence = y.mean()
print("threshold  treat-all  model-before  model-after")
for t in [0.05, 0.10, 0.20, 0.30]:
    treat_all = prevalence - (1 - prevalence) * t / (1 - t)
    print(f"{t:>8.0%}  {treat_all:>9.3f}  {net_benefit(p_before, t):>12.3f}  "
          f"{net_benefit(p_after, t):>11.3f}")

# threshold  treat-all  model-before  model-after
#       5%      0.043         0.053        0.058
#      10%     -0.010         0.028        0.040
#      20%     -0.136         0.002        0.020
#      30%     -0.299        -0.009        0.012

## Exercise

**Goal:** Detect miscalibration at a new site, recalibrate, and judge clinical usefulness with a decision curve.

**Data:** Synthetic sites generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the new-site calibration table and draw a smoothed calibration curve.

In [ ]:
# Your code here


### Step 2

Recalibrate with only the intercept (slope fixed at 1) and with intercept and slope; compare.

In [ ]:
# Your code here


### Step 3

Vary the size of the recalibration sample (200, 500, 2,000) and see how stable the result is.

In [ ]:
# Your code here


### Step 4

Plot full decision curves from 1% to 40%, with 'treat all' and 'treat none', before and after recalibration.

In [ ]:
# Your code here


### Step 5

Simulate a third site where one predictor's effect is weaker, and check whether recalibration alone is enough.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/calibration-and-decision-curves](https://ramkumarmd.com/medical-ai/machine-learning/calibration-and-decision-curves). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.